In [2]:
!pip install pygeohash catboost -q

In [3]:
import pandas as pd
import numpy as np
import pygeohash as pgh

from sklearn.cluster import KMeans
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings("ignore")

In [4]:
train = pd.read_csv("C:/Users/chait/Downloads/e88186124ec611f1/dataset/train.csv")
test = pd.read_csv("C:/Users/chait/Downloads/e88186124ec611f1/dataset/test.csv")

print(train.shape)
print(test.shape)

(77299, 11)
(41778, 10)


In [5]:
for col in [
    "RoadType",
    "Temperature",
    "Weather"
]:

    train[f"{col}_missing"] = (
        train[col].isna().astype(int)
    )

    test[f"{col}_missing"] = (
        test[col].isna().astype(int)
    )

In [6]:
train["RoadType"] = (
    train["RoadType"]
    .fillna("Unknown")
)

test["RoadType"] = (
    test["RoadType"]
    .fillna("Unknown")
)

train["Weather"] = (
    train["Weather"]
    .fillna("Unknown")
)

test["Weather"] = (
    test["Weather"]
    .fillna("Unknown")
)

In [7]:
weather_temp = (
    train.groupby("Weather")
    ["Temperature"]
    .median()
)

global_temp = (
    train["Temperature"]
    .median()
)

In [8]:
def fill_temp(df):

    df["Temperature"] = df.apply(

        lambda row:

        weather_temp.get(
            row["Weather"],
            global_temp
        )

        if pd.isna(
            row["Temperature"]
        )

        else row["Temperature"],

        axis=1
    )

    return df

In [9]:
train = fill_temp(train)
test = fill_temp(test)

In [10]:
print(train.isnull().sum().sum())
print(test.isnull().sum().sum())

0
0


In [11]:
def decode_geo(g):

    lat, lon = pgh.decode(g)

    return pd.Series([lat, lon])

train[["lat","lon"]] = (
    train["geohash"]
    .apply(decode_geo)
)

test[["lat","lon"]] = (
    test["geohash"]
    .apply(decode_geo)
)

In [12]:
for df in [train,test]:

    df["lat2"] = df["lat"]**2

    df["lon2"] = df["lon"]**2

    df["lat_lon"] = (
        df["lat"] * df["lon"]
    )

In [13]:
def time_features(df):

    t = (
        df["timestamp"]
        .str.split(
            ":",
            expand=True
        )
    )

    df["hour"] = (
        t[0].astype(int)
    )

    df["minute"] = (
        t[1].astype(int)
    )

    df["time_slot"] = (

        df["hour"]*4 +

        df["minute"]//15
    )

    df["sin_hour"] = np.sin(
        2*np.pi*df["hour"]/24
    )

    df["cos_hour"] = np.cos(
        2*np.pi*df["hour"]/24
    )

    return df

train = time_features(train)
test = time_features(test)

In [14]:
for i in [2,3,4,5]:

    train[f"geo_prefix_{i}"] = (
        train["geohash"]
        .str[:i]
    )

    test[f"geo_prefix_{i}"] = (
        test["geohash"]
        .str[:i]
    )

In [15]:
freq = (
    train["geohash"]
    .value_counts()
)

train["geo_freq"] = (
    train["geohash"]
    .map(freq)
)

test["geo_freq"] = (
    test["geohash"]
    .map(freq)
    .fillna(0)
)

In [16]:
kmeans = KMeans(
    n_clusters=100,
    random_state=42
)

train["geo_cluster"] = (
    kmeans.fit_predict(
        train[["lat","lon"]]
    )
)

test["geo_cluster"] = (
    kmeans.predict(
        test[["lat","lon"]]
    )
)

In [17]:
def target_encode(
    train_df,
    test_df,
    col,
    target
):

    kf = KFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    train_encoded = np.zeros(
        len(train_df)
    )

    global_mean = (
        train_df[target]
        .mean()
    )

    for tr,val in kf.split(train_df):

        means = (

            train_df.iloc[tr]

            .groupby(col)[target]

            .mean()
        )

        train_encoded[val] = (

            train_df.iloc[val][col]

            .map(means)

            .fillna(global_mean)
        )

    full_means = (

        train_df
        .groupby(col)[target]
        .mean()
    )

    test_encoded = (

        test_df[col]
        .map(full_means)

        .fillna(global_mean)
    )

    return train_encoded,test_encoded

In [18]:
for col in [

    "geohash",

    "timestamp",

    "RoadType",

    "Weather",

    "geo_cluster"

]:

    train[f"{col}_te"],test[f"{col}_te"] = (

        target_encode(
            train,
            test,
            col,
            "demand"
        )
    )

In [19]:
train["target"] = np.log1p(
    train["demand"]
)

In [20]:
drop_cols = [

    "Index",

    "demand",

    "target"
]

features = [

    c for c in train.columns

    if c not in drop_cols
]

In [21]:
cat_features = [

    "geohash",

    "RoadType",

    "Weather",

    "LargeVehicles",

    "Landmarks",

    "timestamp",

    "geo_prefix_2",

    "geo_prefix_3",

    "geo_prefix_4",

    "geo_prefix_5"
]

for col in cat_features:
    train[col] = train[col].astype(str)
    test[col] = test[col].astype(str)

In [22]:
kf = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

oof = np.zeros(
    len(train)
)

preds = np.zeros(
    len(test)
)

In [23]:
for fold,(tr,val) in enumerate(
    kf.split(train)
):

    print(f"\nFold {fold+1}")

    X_tr = train.iloc[tr][features]

    y_tr = train.iloc[tr]["target"]

    X_val = train.iloc[val][features]

    y_val = train.iloc[val]["target"]

    model = CatBoostRegressor(

        iterations=7000,

        learning_rate=0.02,

        depth=10,

        loss_function="RMSE",

        eval_metric="RMSE",

        l2_leaf_reg=8,

        bootstrap_type="Bayesian",

        bagging_temperature=0.2,

        verbose=500
    )

    model.fit(

        X_tr,

        y_tr,

        cat_features=cat_features,

        eval_set=(X_val,y_val),

        use_best_model=True
    )

    oof[val] = (
        model.predict(X_val)
    )

    fold_pred = (
        model.predict(
            test[features]
        )
    )

    preds += (
        fold_pred
        / kf.n_splits
    )

    model.save_model(
        f"catboost_fold_{fold}.cbm"
    )


Fold 1
0:	learn: 0.1072894	test: 0.1054028	best: 0.1054028 (0)	total: 673ms	remaining: 1h 18m 30s
500:	learn: 0.0262438	test: 0.0271025	best: 0.0271025 (500)	total: 3m 50s	remaining: 49m 52s
1000:	learn: 0.0236653	test: 0.0253625	best: 0.0253625 (1000)	total: 7m 59s	remaining: 47m 55s
1500:	learn: 0.0222985	test: 0.0247227	best: 0.0247227 (1500)	total: 12m 8s	remaining: 44m 28s
2000:	learn: 0.0212737	test: 0.0243840	best: 0.0243840 (2000)	total: 16m 13s	remaining: 40m 32s
2500:	learn: 0.0203235	test: 0.0240950	best: 0.0240950 (2500)	total: 20m 19s	remaining: 36m 32s
3000:	learn: 0.0195233	test: 0.0239190	best: 0.0239189 (2995)	total: 24m 23s	remaining: 32m 29s
3500:	learn: 0.0187716	test: 0.0237743	best: 0.0237743 (3500)	total: 28m 24s	remaining: 28m 23s
4000:	learn: 0.0180745	test: 0.0236678	best: 0.0236678 (4000)	total: 32m 28s	remaining: 24m 20s
4500:	learn: 0.0174130	test: 0.0235762	best: 0.0235760 (4497)	total: 36m 27s	remaining: 20m 14s
5000:	learn: 0.0168108	test: 0.0235053	bes

In [24]:
oof_original = np.expm1(oof)

score = r2_score(
    train["demand"],
    oof_original
)

print(score)

0.9583583612793493


In [25]:
preds = np.expm1(preds)

preds = np.clip(
    preds,
    0,
    1
)

In [27]:
!pip install lightgbm -q

In [28]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

import numpy as np

In [29]:
for col in cat_features:

    train[col] = train[col].astype("category")

    test[col] = test[col].astype("category")

In [30]:
kf = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

lgb_oof = np.zeros(len(train))

lgb_test = np.zeros(len(test))

In [31]:
for fold,(tr,val) in enumerate(
    kf.split(train)
):

    print(
        f"\nFold {fold+1}"
    )

    X_train = train.iloc[tr][features]

    y_train = train.iloc[tr]["target"]

    X_valid = train.iloc[val][features]

    y_valid = train.iloc[val]["target"]

    model = LGBMRegressor(

        n_estimators=5000,

        learning_rate=0.01,

        num_leaves=255,

        max_depth=-1,

        subsample=0.8,

        colsample_bytree=0.8,

        objective="regression",

        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    lgb_oof[val] = (
        model.predict(
            X_valid
        )
    )

    lgb_test += (

        model.predict(
            test[features]
        )

        / kf.n_splits
    )


Fold 1
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017030 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3150
[LightGBM] [Info] Number of data points in the train set: 69569, number of used features: 31
[LightGBM] [Info] Start training from score 0.083260

Fold 2
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.016568 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] 

In [32]:
lgb_score = r2_score(
    train["target"],
    lgb_oof
)

print(lgb_score)

0.9540291403240719


In [33]:
cat_pred = np.expm1(preds)

lgb_pred = np.expm1(lgb_test)

In [51]:
from lightgbm import LGBMRegressor
import numpy as np

In [52]:
lgb2 = LGBMRegressor(

    n_estimators=8000,

    learning_rate=0.01,

    num_leaves=127,

    max_depth=10,

    min_child_samples=50,

    subsample=0.90,

    colsample_bytree=0.90,

    reg_alpha=0.5,

    reg_lambda=1.0,

    objective="regression",

    random_state=123,

    verbose=-1
)

In [53]:
lgb2.fit(
    train[features],
    train["target"]
)

,boosting_type,'gbdt'
,num_leaves,127
,max_depth,10
,learning_rate,0.01
,n_estimators,8000
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,50


In [54]:
lgb2_pred = lgb2.predict(
    test[features]
)

lgb2_pred = np.expm1(
    lgb2_pred
)

lgb2_pred = np.clip(
    lgb2_pred,
    0,
    1
)

In [63]:
from lightgbm import LGBMRegressor

In [64]:
lgb3 = LGBMRegressor(

    n_estimators=12000,

    learning_rate=0.005,

    num_leaves=63,

    max_depth=8,

    min_child_samples=100,

    subsample=0.95,

    colsample_bytree=0.95,

    reg_alpha=1.0,

    reg_lambda=2.0,

    objective="regression",

    random_state=999,

    verbose=-1
)

In [65]:
lgb3.fit(
    train[features],
    train["target"]
)

,boosting_type,'gbdt'
,num_leaves,63
,max_depth,8
,learning_rate,0.005
,n_estimators,12000
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,100


In [66]:
lgb3_pred = lgb3.predict(
    test[features]
)

lgb3_pred = np.expm1(
    lgb3_pred
)

lgb3_pred = np.clip(
    lgb3_pred,
    0,
    1
)

In [72]:
from lightgbm import LGBMRegressor

lgb4 = LGBMRegressor(
    n_estimators=15000,
    learning_rate=0.003,
    num_leaves=31,
    max_depth=6,
    min_child_samples=150,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=2.0,
    reg_lambda=3.0,
    random_state=2025,
    verbose=-1
)

lgb4.fit(
    train[features],
    train["target"]
)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.003
,n_estimators,15000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,150


In [73]:
lgb4_pred = np.expm1(
    lgb4.predict(test[features])
)

lgb4_pred = np.clip(
    lgb4_pred,
    0,
    1
)

In [74]:
blend_N = (

    0.05 * cat_pred +

    0.30 * lgb_pred +

    0.25 * lgb2_pred +

    0.20 * lgb3_pred +

    0.20 * lgb4_pred
)

blend_N = np.clip(
    blend_N,
    0,
    1
)

In [75]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": blend_N
})

submission.to_csv(
    "submission_blend_N.csv",
    index=False
)

In [82]:
pseudo_test = test.copy()

pseudo_test["demand"] = blend_N

pseudo_test["target"] = np.log1p(
    pseudo_test["demand"]
)

In [83]:
train_aug = pd.concat(

    [
        train,

        pseudo_test[train.columns]
    ],

    ignore_index=True
)

print(train_aug.shape)

(119077, 36)


In [84]:
pseudo_model = LGBMRegressor(

    n_estimators=10000,

    learning_rate=0.01,

    num_leaves=127,

    max_depth=10,

    subsample=0.9,

    colsample_bytree=0.9,

    random_state=999,

    verbose=-1
)

In [86]:
# Restore categorical dtypes
for col in cat_features:
    train_aug[col] = train_aug[col].astype("category")

In [87]:
pseudo_model.fit(
    train_aug[features],
    train_aug["target"],
    categorical_feature=cat_features
)

,boosting_type,'gbdt'
,num_leaves,127
,max_depth,10
,learning_rate,0.01
,n_estimators,10000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [89]:
pseudo_pred = np.expm1(

    pseudo_model.predict(
        test[features]
    )
)

pseudo_pred = np.clip(
    pseudo_pred,
    0,
    1
)

In [90]:
blend_pseudo = (

    0.70 * blend_N +

    0.30 * pseudo_pred
)

blend_pseudo = np.clip(
    blend_pseudo,
    0,
    1
)

In [92]:
pseudo_test = test.copy()

pseudo_test["target"] = np.log1p(blend_N)

In [93]:
pseudo_weight = 0.3
real_weight = 1.0

sample_weight = np.concatenate([
    np.ones(len(train)) * real_weight,
    np.ones(len(pseudo_test)) * pseudo_weight
])

In [95]:
print("train columns:", len(train.columns))
print("pseudo_test columns:", len(pseudo_test.columns))

print(
    set(train.columns)
    -
    set(pseudo_test.columns)
)

train columns: 36
pseudo_test columns: 35
{'demand'}


In [96]:
pseudo_test = test.copy()

# Add pseudo labels
pseudo_test["demand"] = blend_N

# Target column
pseudo_test["target"] = np.log1p(
    pseudo_test["demand"]
)

In [97]:
print(
    set(train.columns)
    -
    set(pseudo_test.columns)
)

set()


In [98]:
train_aug = pd.concat(
    [
        train,
        pseudo_test.reindex(
            columns=train.columns
        )
    ],
    ignore_index=True
)

In [99]:
print(train_aug.shape)

(119077, 36)


In [100]:
for col in cat_features:
    train_aug[col] = (
        train_aug[col]
        .astype("category")
    )

In [101]:
pseudo_model.fit(
    train_aug[features],
    train_aug["target"],
    sample_weight=sample_weight,
    categorical_feature=cat_features
)

,boosting_type,'gbdt'
,num_leaves,127
,max_depth,10
,learning_rate,0.01
,n_estimators,10000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [102]:
pseudo_pred = pseudo_model.predict(
    test[features]
)

pseudo_pred = np.expm1(
    pseudo_pred
)

pseudo_pred = np.clip(
    pseudo_pred,
    0,
    1
)

print(pseudo_pred[:5])

[0.05920452 0.02428739 0.01197295 0.04141613 0.06888534]


In [103]:
blend_pseudo_15 = (

    0.85 * blend_N +

    0.15 * pseudo_pred
)

blend_pseudo_15 = np.clip(
    blend_pseudo_15,
    0,
    1
)

In [104]:
submission = pd.DataFrame({

    "Index": test["Index"],

    "demand": blend_pseudo_15
})

submission.to_csv(
    "submission_pseudo_15.csv",
    index=False
)

submission.head()

,Index,demand
0,0,0.053318
1,1,0.025066
2,2,0.009162
3,3,0.041120
4,4,0.067098


In [109]:
!pip install xgboost -q

In [110]:
from sklearn.preprocessing import LabelEncoder

train_xgb = train.copy()
test_xgb = test.copy()

for col in cat_features:

    le = LabelEncoder()

    combined = pd.concat([
        train_xgb[col].astype(str),
        test_xgb[col].astype(str)
    ])

    le.fit(combined)

    train_xgb[col] = le.transform(
        train_xgb[col].astype(str)
    )

    test_xgb[col] = le.transform(
        test_xgb[col].astype(str)
    )

In [111]:
from xgboost import XGBRegressor

xgb = XGBRegressor(

    n_estimators=5000,

    learning_rate=0.01,

    max_depth=8,

    subsample=0.8,

    colsample_bytree=0.8,

    objective="reg:squarederror",

    random_state=42
)

xgb.fit(
    train_xgb[features],
    train_xgb["target"]
)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [112]:
xgb_pred = np.expm1(
    xgb.predict(
        test_xgb[features]
    )
)

xgb_pred = np.clip(
    xgb_pred,
    0,
    1
)

In [113]:
blend_xgb = (

    0.90 * blend_pseudo_15 +

    0.10 * xgb_pred
)

blend_xgb = np.clip(
    blend_xgb,
    0,
    1
)

In [122]:
blend_xgb_40 = (
    0.60 * blend_pseudo_15 +
    0.40 * xgb_pred
)

blend_xgb_40 = np.clip(blend_xgb_40,0,1)

In [124]:
pd.DataFrame({
    "Index": test["Index"],
    "demand": blend_xgb_40
}).to_csv(
    "submission_xgb_40.csv",
    index=False
)